In [24]:
!pip install pandas numpy scikit-learn scipy category_encoders


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
from scipy import stats
import category_encoders as ce

In [26]:
df = pd.read_csv('dataset.csv')

print("Dataset Loaded Successfully")
df.head()

Dataset Loaded Successfully


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
print("Missing Values in Dataset:\n")

print(df.isnull().sum())

Missing Values in Dataset:

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [28]:
if 'Age' in df.columns:
    
    mean_imputer = SimpleImputer(strategy='mean')
    
    df['Age'] = mean_imputer.fit_transform(df[['Age']])
    
    print("Mean Imputation Applied on Age Column")

Mean Imputation Applied on Age Column


In [29]:
mode_imputer = SimpleImputer(strategy='most_frequent')

df['Embarked'] = mode_imputer.fit_transform(df[['Embarked']]).ravel()

print("Missing values in Embarked filled using Mode")

Missing values in Embarked filled using Mode


In [30]:
median_imputer = SimpleImputer(strategy='median')

df['Fare'] = median_imputer.fit_transform(df[['Fare']])

print("Missing values in Fare filled using Median")

Missing values in Fare filled using Median


In [31]:
# Select numerical columns from Titanic dataset

numeric_cols = [
    'Age',
    'Fare',
    'SibSp',
    'Parch'
]

# Create KNN Imputer

knn_imputer = KNNImputer(
    n_neighbors=3
)

# Apply KNN Imputation

df[numeric_cols] = knn_imputer.fit_transform(
    df[numeric_cols]
)

print("KNN Imputation Applied Successfully")

df[numeric_cols].head()

KNN Imputation Applied Successfully


,Age,Fare,SibSp,Parch
0,22.0,7.2500,1.0,0.0
1,38.0,71.2833,1.0,0.0
2,26.0,7.9250,0.0,0.0
3,35.0,53.1000,1.0,0.0
4,35.0,8.0500,0.0,0.0


In [32]:
df['Cabin'] = df['Cabin'].fillna('Unknown')

print("Cabin missing values replaced with 'Unknown'")

Cabin missing values replaced with 'Unknown'


In [33]:
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64


In [34]:
z_scores = np.abs(stats.zscore(df['Fare']))

df = df[z_scores < 3]

print("Outliers Removed from Fare Column")

Outliers Removed from Fare Column


In [35]:
Q1 = df['Fare'].quantile(0.25)

Q3 = df['Fare'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[
    (df['Fare'] >= lower_bound) &
    (df['Fare'] <= upper_bound)
]

print("Outliers Removed Using IQR")

Outliers Removed Using IQR


In [36]:
print(df.memory_usage(deep=True))

Index           6176
PassengerId     6176
Survived        6176
Pclass          6176
Name           64432
Sex            47576
Age             6176
SibSp           6176
Parch           6176
Ticket         49221
Fare            6176
Cabin          48969
Embarked       44776
dtype: int64


In [37]:
# Integer columns

for col in df.select_dtypes(include=['int64']).columns:
    
    df[col] = pd.to_numeric(df[col], downcast='integer')


# Float columns

for col in df.select_dtypes(include=['float64']).columns:
    
    df[col] = pd.to_numeric(df[col], downcast='float')


# Object columns to category

for col in df.select_dtypes(include=['object']).columns:
    
    df[col] = df[col].astype('category')

print("Memory Optimization Completed")

Memory Optimization Completed


C:\Users\PATEL PRINS\AppData\Local\Temp\ipykernel_28660\108611937.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [38]:
print(df.memory_usage(deep=True))

Index           6176
PassengerId     1544
Survived         772
Pclass           772
Name           65976
Sex              896
Age             3088
SibSp           3088
Parch           3088
Ticket         41766
Fare            3088
Cabin           6012
Embarked         946
dtype: int64


In [39]:
df_encoded = pd.get_dummies(
    df,
    columns=['Sex', 'Embarked', 'Cabin']
)

print("One-Hot Encoding Applied")

df_encoded.head()

One-Hot Encoding Applied


,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,...,Cabin_F E69,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown
0,1,0,3,"Braund, Mr. Owen Harris",22.000000,1.0,0.0,A/5 21171,7.250000,False,...,False,False,False,False,False,False,False,False,False,True
2,3,1,3,"Heikkinen, Miss. Laina",26.000000,0.0,0.0,STON/O2. 3101282,7.925000,True,...,False,False,False,False,False,False,False,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.000000,1.0,0.0,113803,53.099998,True,...,False,False,False,False,False,False,False,False,False,False
4,5,0,3,"Allen, Mr. William Henry",35.000000,0.0,0.0,373450,8.050000,False,...,False,False,False,False,False,False,False,False,False,True
5,6,0,3,"Moran, Mr. James",29.699118,0.0,0.0,330877,8.458300,False,...,False,False,False,False,False,False,False,False,False,True


In [40]:
encoder = ce.TargetEncoder(cols=['Pclass'])

df['Pclass'] = encoder.fit_transform(
    df['Pclass'],
    df['Survived']
)

print("Target Encoding Applied on Pclass")

df.head()

Target Encoding Applied on Pclass


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,0.245868,"Braund, Mr. Owen Harris",male,22.000000,1.0,0.0,A/5 21171,7.250000,Unknown,S
2,3,1,0.245868,"Heikkinen, Miss. Laina",female,26.000000,0.0,0.0,STON/O2. 3101282,7.925000,Unknown,S
3,4,1,0.504486,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1.0,0.0,113803,53.099998,C123,S
4,5,0,0.245868,"Allen, Mr. William Henry",male,35.000000,0.0,0.0,373450,8.050000,Unknown,S
5,6,0,0.245868,"Moran, Mr. James",male,29.699118,0.0,0.0,330877,8.458300,Unknown,Q


In [41]:
df_encoded.to_csv(
    'processed_titanic_dataset.csv',
    index=False
)

print("Processed Dataset Saved Successfully")

Processed Dataset Saved Successfully


In [42]:
df_encoded.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,...,Cabin_F E69,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown
0,1,0,3,"Braund, Mr. Owen Harris",22.000000,1.0,0.0,A/5 21171,7.250000,False,...,False,False,False,False,False,False,False,False,False,True
2,3,1,3,"Heikkinen, Miss. Laina",26.000000,0.0,0.0,STON/O2. 3101282,7.925000,True,...,False,False,False,False,False,False,False,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.000000,1.0,0.0,113803,53.099998,True,...,False,False,False,False,False,False,False,False,False,False
4,5,0,3,"Allen, Mr. William Henry",35.000000,0.0,0.0,373450,8.050000,False,...,False,False,False,False,False,False,False,False,False,True
5,6,0,3,"Moran, Mr. James",29.699118,0.0,0.0,330877,8.458300,False,...,False,False,False,False,False,False,False,False,False,True


In [43]:
new_df = pd.read_csv(
    'processed_titanic_dataset.csv'
)

new_df.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,...,Cabin_F E69,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown
0,1,0,3,"Braund, Mr. Owen Harris",22.000000,1.0,0.0,A/5 21171,7.2500,False,...,False,False,False,False,False,False,False,False,False,True
1,3,1,3,"Heikkinen, Miss. Laina",26.000000,0.0,0.0,STON/O2. 3101282,7.9250,True,...,False,False,False,False,False,False,False,False,False,True
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.000000,1.0,0.0,113803,53.1000,True,...,False,False,False,False,False,False,False,False,False,False
3,5,0,3,"Allen, Mr. William Henry",35.000000,0.0,0.0,373450,8.0500,False,...,False,False,False,False,False,False,False,False,False,True
4,6,0,3,"Moran, Mr. James",29.699118,0.0,0.0,330877,8.4583,False,...,False,False,False,False,False,False,False,False,False,True


In [44]:
import pandas as pd

output_df = pd.read_csv(
    'processed_titanic_dataset.csv'
)

print("Output Dataset Loaded Successfully")

Output Dataset Loaded Successfully


In [45]:
pd.set_option('display.max_rows', None)

pd.set_option('display.max_columns', None)

display(output_df)

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Cabin_A10,Cabin_A14,Cabin_A16,Cabin_A19,Cabin_A20,Cabin_A23,Cabin_A24,Cabin_A26,Cabin_A31,Cabin_A32,Cabin_A36,Cabin_A5,Cabin_A6,Cabin_A7,Cabin_B102,Cabin_B18,Cabin_B19,Cabin_B20,Cabin_B30,Cabin_B37,Cabin_B38,Cabin_B39,Cabin_B4,Cabin_B42,Cabin_B50,Cabin_B51 B53 B55,Cabin_B71,Cabin_B94,Cabin_C101,Cabin_C103,Cabin_C104,Cabin_C106,Cabin_C110,Cabin_C111,Cabin_C118,Cabin_C123,Cabin_C124,Cabin_C126,Cabin_C128,Cabin_C148,Cabin_C30,Cabin_C47,Cabin_C49,Cabin_C52,Cabin_C87,Cabin_C90,Cabin_D,Cabin_D17,Cabin_D19,Cabin_D21,Cabin_D28,Cabin_D30,Cabin_D35,Cabin_D45,Cabin_D46,Cabin_D47,Cabin_D50,Cabin_D56,Cabin_D6,Cabin_E10,Cabin_E101,Cabin_E12,Cabin_E121,Cabin_E17,Cabin_E24,Cabin_E25,Cabin_E31,Cabin_E33,Cabin_E36,Cabin_E38,Cabin_E44,Cabin_E46,Cabin_E50,Cabin_E58,Cabin_E63,Cabin_E77,Cabin_E8,Cabin_F E69,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown
0,1,0,3,"Braund, Mr. Owen Harris",22.000000,1.0,0.0,A/5 21171,7.2500,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
1,3,1,3,"Heikkinen, Miss. Laina",26.000000,0.0,0.0,STON/O2. 3101282,7.9250,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.000000,1.0,0.0,113803,53.1000,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,5,0,3,"Allen, Mr. William Henry",35.000000,0.0,0.0,373450,8.0500,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4,6,0,3,"Moran, Mr. James",29.699118,0.0,0.0,330877,8.4583,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False